**Solar Panal Defect Detection Using Deeplearning.**

In [ ]:
# Train the model using MobileNetV2.

from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input           # type: ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator                            # type: ignore 
from tensorflow.keras import layers, models                                                    # type: ignore
from sklearn.metrics import classification_report
import tensorflow as tf
import numpy as np

dataset = r"/Users/joe/Downloads/Faulty_solar_panel"                # Dataset path.

# Train generator. 
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    horizontal_flip=True)


# Validation generator. 
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                        validation_split=0.2)


train_data = train_datagen.flow_from_directory(dataset,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True)

val_data = val_datagen.flow_from_directory(dataset,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False)

base_model = MobileNetV2(input_shape=(224,224,3),include_top=False,weights='imagenet')

base_model.trainable = False               # Freeze the base model.

model_mobilenet = models.Sequential([base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(), 
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(6, activation='softmax')])

class_weights = {0: 1.3, 1: 1.0, 2: 1.1, 3: 1.8, 4: 2.5, 5: 1.0  }

model_mobilenet.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

model_mobilenet.fit(train_data, validation_data=val_data, epochs=20, class_weight=class_weights)


23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 404ms/step - accuracy: 0.9309 - loss: 0.2373 - val_accuracy: 0.7644 - val_loss: 0.6986
Epoch 19/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 403ms/step - accuracy: 0.9097 - loss: 0.3997 - val_accuracy: 0.7644 - val_loss: 0.6822
Epoch 20/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 430ms/step - accuracy: 0.9126 - loss: 0.3855 - val_accuracy: 0.8103 - val_loss: 0.6627


In [15]:
# Classification report for mobilenet.

y_pred_probs = model_mobilenet.predict(val_data)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = val_data.classes

class_names = list(val_data.class_indices.keys())

report = classification_report(y_true, y_pred, target_names=class_names)
print("\nClassification Report:\n")
print(report)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 235ms/step

Classification Report:

                   precision    recall  f1-score   support

        Bird-drop       0.85      0.68      0.76        41
            Clean       0.85      0.87      0.86        38
            Dusty       0.65      0.89      0.76        38
Electrical-damage       0.90      0.95      0.93        20
  Physical-Damage       1.00      0.38      0.56        13
     Snow-Covered       0.92      0.92      0.92        24

         accuracy                           0.81       174
        macro avg       0.86      0.78      0.79       174
     weighted avg       0.83      0.81      0.80       174



**ResNet50 Model Training.**

In [ ]:
# Train the model using pretrained ResNet50.

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input                  # type: ignore

# Train generator. 
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    horizontal_flip=True)


# Validation generator. 
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                        validation_split=0.2)


train_data = train_datagen.flow_from_directory(dataset,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True)

val_data = val_datagen.flow_from_directory(dataset,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False)

base_model = ResNet50(weights='imagenet',include_top=False,input_shape=(224, 224, 3))

base_model.trainable = False              # Freeze base model

model_res = models.Sequential([base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(6, activation='softmax')  ])

class_weights = {0: 1.0, 1: 1.0, 2: 1.2, 3: 1.5, 4: 3.0, 5: 1.0}

model_res.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

model_res.fit(train_data, validation_data=val_data, epochs=20 ,class_weight=class_weights)

23/23 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.9365 - loss: 0.1964 - val_accuracy: 0.8276 - val_loss: 0.6147
Epoch 18/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 23s 986ms/step - accuracy: 0.9196 - loss: 0.2383 - val_accuracy: 0.8448 - val_loss: 0.6207
Epoch 19/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.9577 - loss: 0.1352 - val_accuracy: 0.8391 - val_loss: 0.7362
Epoch 20/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.9365 - loss: 0.2574 - val_accuracy: 0.8793 - val_loss: 0.5328


In [3]:
# Classification report for resnet50.

y_pred_probs = model_res.predict(val_data)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = val_data.classes

class_names = list(val_data.class_indices.keys())

report = classification_report(y_true, y_pred, target_names=class_names)
print("\nClassification Report:\n")
print(report)

6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 580ms/step

Classification Report:

                   precision    recall  f1-score   support

        Bird-drop       0.88      0.88      0.88        41
            Clean       0.82      0.84      0.83        38
            Dusty       0.84      0.84      0.84        38
Electrical-damage       0.90      0.95      0.93        20
  Physical-Damage       0.91      0.77      0.83        13
     Snow-Covered       1.00      1.00      1.00        24

         accuracy                           0.88       174
        macro avg       0.89      0.88      0.89       174
     weighted avg       0.88      0.88      0.88       174



**EfficientNetB0 Model Training.**

In [ ]:
# Train the model using pretrained EfficientNet.

from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input         # type: ignore

# Train generator. 
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    horizontal_flip=True)


# Validation generator. 
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                        validation_split=0.2)


train_data = train_datagen.flow_from_directory(dataset,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True)

val_data = val_datagen.flow_from_directory(dataset,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False)

base_model = EfficientNetB0(weights='imagenet',include_top=False,input_shape=(224,224,3))

base_model.trainable = False

model_eff = models.Sequential([base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_data.num_classes, activation='softmax')])

class_weights = {0: 1.2, 1: 1.0, 2: 0.9, 3: 1.3, 4: 1.9, 5: 1.0 }

model_eff.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

model_eff.fit(train_data,validation_data=val_data,epochs=20,class_weight=class_weights)


23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 551ms/step - accuracy: 0.9535 - loss: 0.1725 - val_accuracy: 0.8563 - val_loss: 0.5178
Epoch 19/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 539ms/step - accuracy: 0.9464 - loss: 0.1875 - val_accuracy: 0.8678 - val_loss: 0.5455
Epoch 20/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 541ms/step - accuracy: 0.9408 - loss: 0.2075 - val_accuracy: 0.8333 - val_loss: 0.6363


In [19]:
# Classification report for efficientnet.

y_pred_probs = model_eff.predict(val_data)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = val_data.classes

class_names = list(val_data.class_indices.keys())

report = classification_report(y_true, y_pred, target_names=class_names)
print("\nClassification Report:\n")
print(report)

6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 329ms/step

Classification Report:

                   precision    recall  f1-score   support

        Bird-drop       0.91      0.76      0.83        41
            Clean       0.87      0.71      0.78        38
            Dusty       0.64      0.92      0.75        38
Electrical-damage       0.91      1.00      0.95        20
  Physical-Damage       1.00      0.69      0.82        13
     Snow-Covered       1.00      0.96      0.98        24

         accuracy                           0.83       174
        macro avg       0.89      0.84      0.85       174
     weighted avg       0.86      0.83      0.84       174



In [4]:
# Save the trained model.

model_res.save("ResNet_model.keras")

print("Model saved successfully!")

Model saved successfully!
